# OPSD Data Exploration
This notebook sets up the Python environment, downloads the Open Power System Data hourly dataset for Germany, explores the dataset, and visualizes hourly demand patterns over one week and one month.

## 1. Install Required Libraries
Install `pandas`, `scikit-learn`, `xgboost`, and `streamlit`. Note that `tensorflow` may not be available on Python 3.14 in this environment.

In [ ]:
!pip install pandas scikit-learn xgboost streamlit

In [ ]:
import sys
print('Python version:', sys.version)
try:
    import tensorflow as tf
    print('TensorFlow version:', tf.__version__)
except Exception as exc:
    print('TensorFlow not available in this environment:', exc)

## 2. Download OPSD Dataset
Download the OPSD hourly time series dataset and save it to the repository `data/` folder.

In [ ]:
import requests
from pathlib import Path

data_dir = Path('..') / 'data'
data_dir.mkdir(parents=True, exist_ok=True)
output_file = data_dir / 'opsd_time_series_60min.csv'
candidate_dates = [
    '2025-04-30', '2025-03-31', '2025-02-28', '2025-01-31',
    '2024-12-31', '2024-11-30', '2024-10-31', '2024-09-30'
]
downloaded = False
for date in candidate_dates:
    url = f'https://data.open-power-system-data.org/time_series/{date}/time_series_60min.csv'
    print('Checking', url)
    try:
        head = requests.head(url, allow_redirects=True, timeout=20)
        if head.status_code == 200:
            response = requests.get(url, timeout=120)
            response.raise_for_status()
            output_file.write_bytes(response.content)
            print('Downloaded OPSD dataset to', output_file)
            downloaded = True
            break
        else:
            print('Not found:', head.status_code)
    except Exception as exc:
        print('Failed for', url, exc)

if not downloaded:
    raise RuntimeError('Could not download OPSD dataset. Please verify the URL or network access.')

## 3. Load and Explore Data
Load the downloaded dataset into pandas, inspect the shape, missing values, date range, and column types.

In [ ]:
import pandas as pd
from pathlib import Path

data_file = Path('..') / 'data' / 'opsd_time_series_60min.csv'
print('Data file exists:', data_file.exists())
sample = pd.read_csv(data_file, sep=';', nrows=5)
print('Sample columns:', sample.columns.tolist())

df = pd.read_csv(data_file, sep=';', low_memory=False)
df.columns = df.columns.str.strip()
if 'utc_timestamp' in df.columns:
    df['utc_timestamp'] = pd.to_datetime(df['utc_timestamp'], utc=True, errors='coerce')
    df = df.set_index('utc_timestamp').sort_index()
else:
    print('Warning: expected utc_timestamp column not found.')

print('Shape:', df.shape)
print('Date range:', df.index.min(), 'to', df.index.max())
print('Missing values by column:')
print(df.isna().sum().loc[lambda x: x > 0])
print('\nData types:')
print(df.dtypes)

## 4. Plot Hourly Demand Over 1 Week
Select a one-week period from the dataset and plot hourly demand to highlight morning peaks, night dips, and weekend patterns.

In [ ]:
import matplotlib.pyplot as plt

load_cols = [c for c in df.columns if 'load' in c.lower()]
if load_cols:
    demand_col = load_cols[0]
else:
    demand_col = df.columns[0]
print('Using demand column:', demand_col)

one_week_start = df.index.min()
one_week = df.loc[one_week_start:one_week_start + pd.Timedelta(days=7)]

plt.figure(figsize=(14, 5))
plt.plot(one_week.index, one_week[demand_col], label='Hourly demand')
plt.title('Hourly Demand Over One Week')
plt.xlabel('Date')
plt.ylabel(demand_col)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.legend()
plt.show()

## 5. Plot Hourly Demand Over 1 Month
Plot a one-month period of hourly demand to observe broader cycles, weekends, and daily peak shapes.

In [ ]:
one_month = df.loc[one_week_start:one_week_start + pd.Timedelta(days=30)]

plt.figure(figsize=(16, 6))
plt.plot(one_month.index, one_month[demand_col], label='Hourly demand')
plt.title('Hourly Demand Over One Month')
plt.xlabel('Date')
plt.ylabel(demand_col)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.legend()
plt.show()

## 6. Set Up GitHub Repository Structure
Create the repo folders for data, notebooks, models, and dashboard structure.

In [ ]:
from pathlib import Path

root = Path('..')
for folder in ['data', 'notebooks', 'models', 'dashboard']:
    path = root / folder
    path.mkdir(parents=True, exist_ok=True)
    print('Ensured', path)

print('Repository folder structure is ready.')